# Tutorial 1: Basics and first contact with Inspect AI

Welcome to the first tutorial in our AI Safety Evaluations course.

**What you'll learn:**

- Connect Inspect AI to a language model (locally via Ollama, or via cloud API)
- Run your first evaluation
- Understand how tasks are structured: dataset → solver → scorer
- View and analyze results with `inspect view`
- Create single choice and multiple choice benchmarks
- Analyzing position bias in multiple-choice tasks

**By the end:** You'll have a working evaluation pipeline and understand how to build your own benchmarks.

---
## Prerequisites: Model setup

> **💡 Inspect AI only needs a model name** — the model itself can come from anywhere.

**In this tutorial, we'll use Ollama and Perplexity, SambaNova as examples**, but you can substitute any provider: OpenAI, Anthropic, Google, local inference servers, or any OpenAI-compatible endpoint.

**Cost note:** Cloud APIs have small free tiers and then charge per token. Local models (Ollama) are completely free. For this course, a local model is sufficient for all assignments.

See [Inspect AI models docs](https://inspect.ai-safety-institute.org.uk/models.html) for the full list.

---
## Part 1: Local environment setup (Ollama)

Running evaluations locally gives you complete control and privacy.

### 1.1. Installing Ollama

**Before running this notebook, install Ollama:**

**macOS:** `brew install ollama` or download from https://ollama.ai/download

**Linux:**
```bash
curl -fsSL https://ollama.ai/install.sh | sh
```

**Windows:** download from https://ollama.ai/download

After installation, start the server and download a model:
```bash
ollama serve
ollama pull llama2        # or use deepseek-r1:1.5b for a smaller option
```

### 1.2. Check Ollama connection

In [ ]:
import sys
!{sys.executable} -m pip install requests inspect-ai openai

In [ ]:
import sys
!{sys.executable} -m pip install ipywidgets

In [3]:
import requests

def check_ollama():
    """Check Ollama connection and show installed models."""
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        if response.status_code != 200:
            print("❌ Ollama returned an error")
            return False
            
        models = response.json().get('models', [])
        total_size = sum(m['size'] for m in models)
        
        print("✅ Ollama is running!")
        print(f"\n📊 Installed models: {len(models)} ({total_size / 1e9:.1f} GB total)")
        
        for m in models:
            print(f"   - {m['name']}: {m['size'] / 1e9:.2f} GB")
        
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to Ollama")
        print("   Start it with: ollama serve")

check_ollama()

✅ Ollama is running!

📊 Installed models: 5 (20.7 GB total)
   - qwen2:7b-instruct-q4_K_M: 4.68 GB
   - qwen2:1.5b-instruct-q4_K_M: 0.99 GB
   - llama2:13b: 7.37 GB
   - llama2:7b: 3.83 GB
   - llama2:latest: 3.83 GB


---
## Part 2: Basic Inspect setup

### 2.1. Install Inspect AI

In [4]:
!pip install -q inspect-ai openai
print("✅ Installed: inspect-ai, openai")



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✅ Installed: inspect-ai, openai


## Assignment 1: 'Hello world' in eval

Let's create the simplest possible evaluation!

**To do:** add one more `Sample()` to the dataset.

In [3]:
from inspect_ai import Task, task, eval
from inspect_ai.dataset import Sample
from inspect_ai.scorer import exact, match, model_graded_fact, choice, pattern
from inspect_ai.solver import (
    generate, system_message, chain_of_thought, 
    prompt_template, multiple_choice
)

In [4]:
@task
def hello_model():
    """Test your model setup with simple questions."""
    return Task(
        dataset=[
            Sample(
                input="Say 'Hello world!' and nothing else.",
                target="Hello world!"
            ),
            Sample(
                input="2+2=",
                target="4"
            ),
            Sample(
                input="What is the surname of Sheldon from The Big Bang Theory?",
                target="Cooper"
            ),
            Sample(
                input="What color is the sky on a clear day?",
                target="blue"
            )
        ],
        solver=[generate()],
        scorer=match(
            location="end",
            ignore_case=True,
            numeric=False
        )
    )

**Run the evaluation:**

This will take a minute or two depending on your hardware.

In [6]:
eval(
    hello_model,
    model="ollama/llama2",
    # limit=1  # Uncomment to test with just 1 sample
)

Output()

### Выводы:

##### Первый eval успешно запустился. Accuracy = 0.75, потому что 1 из 4 sample не был засчитан scorer’ом. При этом ошибка была не в знании модели, а в формате ответа: модель ответила содержательно верно, но слишком развернуто. 

##### Также он показал, что Inspect AI работает, но вместе с тем сразу продемонстрировал ограничение evals: итоговая метрика зависит не только от реальных знаний модели, но и от того, как сформулирован prompt и как устроен scorer.

---
## Part 3: API setup (optional)

If you don't have a GPU or want to test cloud models, you can use API providers.

### 3.1. Perplexity setup

1. Get an API key at https://www.perplexity.ai/settings/api
2. Set it in the cell below

In [ ]:
import os
YOUR_API_KEY = ''
os.environ['PERPLEXITY_API_KEY'] = YOUR_API_KEY  # Replace with your key

In [ ]:
eval(
    hello_model,
    model="perplexity/sonar",
    # limit=1  # Uncomment to test with just 1 sample
)

## Sambanova 

Get API KEY https://cloud.sambanova.ai/apis here 

In [ ]:
# !pip install SambaNova

In [ ]:
from sambanova import SambaNova

YOUR_API_KEY = ''
os.environ['SAMBANOVA_API_KEY'] = YOUR_API_KEY  # Replace with your key

In [ ]:
eval(
    hello_model,
    model="sambanova/DeepSeek-V3.1",
    # limit=1  # Uncomment to test with just 1 sample
)

---
## Part 4: Viewing results with Inspect view

Every evaluation saves a log file. `inspect view` opens a web UI to explore them.

### 4.1. Launch Inspect view

1. In terminal, from the notebook's folder, run: `inspect view`
2. Open in browser: http://localhost:7575

This will:
1. Show all evaluation logs in an interactive interface
2. Allow you to drill down into individual samples

**Alternative options:**

```bash
# View logs from a specific directory
inspect view --log-dir ./experiment-logs

# Use a different port
inspect view --port 8080
```

**Troubleshooting:**

- If `inspect: command not found` → try `python -m inspect_ai view`
- If the page won't load → check that you're in the correct folder (logs are saved relative to where you run evaluations)

In [ ]:
# # Uncomment and run this cell if inspect view shows no logs

# import os
# from pathlib import Path
# 
# log_files = list(Path("./logs").glob("*.eval")) if Path("./logs").exists() else []
# 
# if not log_files:
#     print("❌ No log files found")
#     print("   Run at least one eval() in this notebook first")
#     print(f"   Current directory: {os.getcwd()}")
# else:
#     print(f"✅ Found {len(log_files)} log file(s):")
#     for f in sorted(log_files)[-5:]:  # show last 5
#         print(f"   {f.name}")

### Assignment 2: Explore your logs

In the Inspect view UI you can:

- **See overall accuracy** for each evaluation run
- **Click on individual samples** to see the model's response
- **Compare runs** with different models or parameters
- **Filter by metadata** (e.g., show only "hard" problems)
- **Export results** for further analysis

**Tip:** Keep `inspect view` running in a separate terminal while you work through this notebook. It auto-refreshes when new evaluations complete.

---
## Part 5: Understanding benchmark structure

### 5.1. Task components overview

Every Inspect `Task` consists of:

```
Task {
    dataset: [Sample, Sample, ...],    # data to evaluate on
    solver: [Solver, Solver, ...],     # how to process
    scorer: Scorer,                    # how to score
    **parameters                       
}
```

**Component flow:**
```
Dataset (Samples) → Solver(s) → Model → Scorer → Results
```

### 5.2. Sample structure

A Sample contains input/target pairs with optional metadata:

In [7]:
# Example of a fully-featured Sample
sample_example = Sample(
    input="Question or prompt",
    target="Expected answer",
    id="unique_id",
    choices=["Option A", "Option B", "Option C"],  # For multiple choice
    metadata={
        "category": "math",
        "difficulty": "hard"
    }
)

print("Sample components:")
print(f"  - input: {sample_example.input}")
print(f"  - target: {sample_example.target}")
print(f"  - choices: {sample_example.choices}")
print(f"  - metadata: {sample_example.metadata}")

Sample components:
  - input: Question or prompt
  - target: Expected answer
  - choices: ['Option A', 'Option B', 'Option C']
  - metadata: {'category': 'math', 'difficulty': 'hard'}


---
## Part 6: Understanding solvers

### 6.1. What is a solver?

A **solver** is a function that transforms a **TaskState** (the prompt + conversation history) and optionally calls the model to generate a response.

**Think of solvers as middleware that:**
1. Modifies the prompt (prompt engineering)
2. Calls the model (generation)
3. Processes the response (extraction, critique, etc.)

### 6.2. The solver pipeline

Solvers are chained together in a pipeline:

```
Input Sample
    ↓
[Solver 1: system_message]
    ↓
[Solver 2: prompt_template]
    ↓
[Solver 3: chain_of_thought]
    ↓
[Solver 4: generate]
    ↓
Model Output → Scorer → Final Result
```

Each solver receives the TaskState, modifies it, and passes it to the next solver.

### 6.3. TaskState - the core data structure

Every solver operates on a **TaskState** containing:

```
TaskState {
    messages: list[ChatMessage],  # Conversation history
    output: ModelOutput,          # Final model output
    user_prompt: str,             # Current user prompt
    input_text: str,              # Original input
    metadata: dict,               # Sample metadata
    choices: list[str],           # For multiple choice
    model: ModelName,             # Current model
    sample_id: int | str,         # Sample identifier
}
```

---
## Part 7: Built-in solvers

**system_message**
```python
system_message(
    message: str        # REQUIRED - the system prompt
)
```

**prompt_template**
```python
prompt_template(
    template: str       # REQUIRED - use {prompt} as placeholder
)
```

**chain_of_thought**
```python
chain_of_thought(
    template: str = None   # optional - custom CoT prompt (default: "Let's think step by step")
)
```

**generate**
```python
generate(
    max_tokens: int = None,      # optional - limit response length
    temperature: float = None,   # optional - 0.0 = deterministic, 1.0 = creative
    top_p: float = None,         # optional - nucleus sampling
    stop_seqs: list[str] = None  # optional - stop generation at these strings
)
```

**multiple_choice**
```python
multiple_choice(
    cot: bool = False,              # optional - add chain-of-thought
    multiple_correct: bool = False, # optional - allow multiple answers
    shuffle: bool = False           # optional - randomize choice order
)
```

**Typical pipeline:**
```
system_message → prompt_template → chain_of_thought → generate
```

`multiple_choice()` replaces the entire chain - it handles prompting and generation internally.

**Viewing solver execution:** in `inspect view`, click any sample → messages tab shows each solver's contribution.

### 7.1 system_message()

**Purpose:** prepend a system role message to guide model behavior.

**When to use:**
- establish the model's role or persona
- set global guidelines or constraints
- define the evaluation context


In [8]:
@task
def example_system_message():
    """
    Demonstrates system_message() solver.
    The system prompt tells the model to be concise.
    """
    return Task(
        dataset=[
            Sample(input="What is 15 * 8?", target="120"),
            Sample(input="What is 99 + 1?", target="100"),
        ],
        solver=[
            system_message("You are a calculator. Reply with only the number, nothing else."),
            generate()
        ],
        scorer=match(numeric=True),
    )

# Run and check the Messages tab in inspect view
eval(example_system_message, model="ollama/llama2")

Output()

### Выводы:

##### Эта ячейка демонстрирует работу system_message() в solver: перед генерацией ответа модели добавляется системная инструкция, которая должна сделать ответы более краткими и формально правильными. 

### 7.2 prompt_template()

**Purpose:** substitute variables into a template to reformat prompts.

**When to use:**
- add specific output format requirements
- include examples or demonstrations
- structure prompts consistently
- add reasoning steps or breakdowns


In [9]:
STEP_BY_STEP_TEMPLATE = '''
Solve this problem step by step:

Problem: {prompt}

Structure:
1. Understand the problem
2. Plan your approach
3. Solve it
4. Final answer format: ANSWER: <value>
'''.strip()

@task
def example_prompt_template():
    """
    Demonstrates prompt_template() solver.
    The template adds structure to the prompt.
    """
    return Task(
        dataset=[
            Sample(input="What is 25 * 4?", target="100"),
            Sample(input="What is 144 / 12?", target="12"),
        ],
        solver=[
            system_message("You are a math tutor."),
            prompt_template(STEP_BY_STEP_TEMPLATE),
            generate()
        ],
        scorer=match(numeric=True),
    )

# Run and see how the template structures the prompt
eval(example_prompt_template, model="ollama/llama2")

Output()

### Выводы:

##### В evals важно не просто спросить модель, а спросить её одинаково каждый раз. Шаблон как раз помогает убрать случайность в формулировке. 

##### prompt_template применяют, когда нужно подавать задачи модели в едином, контролируемом формате. Это помогает стандартизировать eval, управлять формой ответа и иногда лучше выявлять реальные способности модели.

### 7.3 chain_of_thought()

**Purpose:** ask the model to "think step by step" before answering.

**When to use:**
- math and logic problems
- multi-step reasoning tasks
- when you want to see the model's thought process

In [10]:
@task
def example_chain_of_thought():
    """
    Demonstrates chain_of_thought() solver.
    Compare accuracy with and without CoT in inspect view.
    """
    return Task(
        dataset=[
            Sample(
                input="If Alice has 3 apples and Bob gives her 2 more, how many does she have?",
                target="5"
            ),
            Sample(
                input="A train travels 100 km in 2 hours. At this rate, how far in 5 hours?",
                target="250"
            ),
        ],
        solver=[
            system_message("Solve the problem. End with: ANSWER: <number>"),
            chain_of_thought(),
            generate()
        ],
        scorer=match(numeric=True),
    )

eval(example_chain_of_thought, model="ollama/llama2")

Output()

### Выводы:

##### В первом запуске ответ был засчитан как неправильный, потому что модель действительно неверно решила задачу. Она смешала единицы измерения: вычислила скорость в км/мин, но затем умножила её на 5 часов вместо 300 минут. Поэтому вместо правильного ответа 250 получила 41.5. Scorer лишь зафиксировал эту ошибку. Во втором запуске модель решила задачу верно.

##### Пример с chain-of-thought показывает, что добавление пошагового рассуждения — это способ elicitation, который может помочь модели решать задачу, но не гарантирует правильный результат. В Inspect View такой пример полезен тем, что позволяет увидеть не только факт ошибки, но и место, где логика модели сломалась.

### 7.5. multiple_choice()

Special solver for A/B/C/D questions. Handles formatting and answer extraction automatically.

**When to use:**
- multiple choice questions (use instead of generate)
- must have letter target: "A", "B", "C", etc.

**Note:** When using `multiple_choice()`, use `choice()` as the scorer.

In [12]:
@task
def example_multiple_choice_with_cot():
    """
    Demonstrates multiple_choice(cot=True).
    Model reasons before selecting an answer.
    """
    return Task(
        dataset=[
            Sample(
                input="Light travels faster than sound. If you see lightning and hear thunder 3 seconds later, approximately how far away was the strike?",
                choices=["100 meters", "1 kilometer", "3 kilometers", "10 kilometers"],
                target="B"  # ~1 km (sound travels ~340 m/s)
            ),
        ],
        solver=multiple_choice(cot=True),
        scorer=choice(),
    )

eval(example_multiple_choice_with_cot, model="ollama/llama2")

Output()

### 7.6 Other solvers

**self_critique()** - Have model refine its own answer
```python
solver=[generate(), self_critique()]
```

**use_tools()** - Enable tool/function calling
```python
solver=[use_tools(calculator()), generate()]
```

---
## Part 8: Single choice tasks

Single choice tasks present the model with limited options to select from.

### 8.1. Simple yes/no classification

The simplest single choice - binary classification:

In [13]:
@task
def yes_no_classification():
    return Task(
        dataset=[
            Sample(
                input="Is Python a programming language?",
                target="Yes"
            ),
            Sample(
                input="Is water dry?",
                target="No"
            ),
            Sample(
                input="Is the Earth round?",
                target="Yes"
            ),
        ],
        solver=[
            system_message("Answer 'Yes' or 'No'. Be concise."),
            generate()
        ],
        scorer=exact(),
    )

eval(
    yes_no_classification,
    model="ollama/llama2"
)

Output()

### 8.2. Multi-class classification

In multi-class classification, the model must choose from 3+ categories. This is common for:
- sentiment analysis (positive / negative / neutral)
- topic classification (sports / politics / tech / ...)
- intent detection (question / command / statement)

---

### Task 2: Build a sentiment classifier

**Your goal:** Create a sentiment classification task with at least 4 samples.

**Note:**
- `system_message` defines the classes and output format
- `target` must exactly match one of your class labels

In [14]:
@task
def sentiment_classification():
    return Task(
        dataset=[
            Sample(
                input="I absolutely loved this movie, it was amazing.",
                target="positive"
            ),
            Sample(
                input="This was the worst restaurant experience of my life.",
                target="negative"
            ),
            Sample(
                input="The meeting is scheduled for 3 PM tomorrow.",
                target="neutral"
            ),
            Sample(
                input="The product works fine, nothing special but no major problems.",
                target="neutral"
            ),
            Sample(
                input="The customer support team was very helpful and kind.",
                target="positive"
            ),
            Sample(
                input="I am very disappointed with the quality of this phone.",
                target="negative"
            ),
        ],
        solver=[
            system_message(
                "You are a sentiment classifier. "
                "Classify the text into exactly one of these labels: "
                "positive, negative, neutral. "
                "Reply with only one label and nothing else."
            ),
            generate()
        ],
        scorer=match(ignore_case=True)
    )

eval(
    sentiment_classification,
    model="ollama/llama2"
)

Output()

### Выводы:

##### Можно предположить что, локальная llama2 в этой постановке лучше распознаёт полярные эмоции, чем нейтральный класс. 

##### Модель достигла accuracy 0.667 (4/6). Она корректно классифицировала все явно positive и negative примеры, но ошиблась на двух neutral примерах, оба раза отнеся их к positive. Это показывает, что в данной постановке локальная llama2 лучше распознаёт ярко выраженную эмоциональную окраску, чем нейтральные или пограничные высказывания. Ошибки были не в формате ответа, а в самой классификации.

In [15]:
@task
def sentiment_classification():
    return Task(
        dataset=[
            Sample(
                input="I absolutely loved this movie, it was amazing.",
                target="positive"
            ),
            Sample(
                input="This was the worst restaurant experience of my life.",
                target="negative"
            ),
            Sample(
                input="The meeting is scheduled for 3 PM tomorrow.",
                target="neutral"
            ),
            Sample(
                input="The product works fine, nothing special but no major problems.",
                target="neutral"
            ),
            Sample(
                input="The customer support team was very helpful and kind.",
                target="positive"
            ),
            Sample(
                input="I am very disappointed with the quality of this phone.",
                target="negative"
            ),
        ],
        solver=[
            system_message(
                "You are a sentiment classifier. "
                "Classify the text into exactly one of these labels: "
                "positive, negative, neutral. "
                "Reply with only one label and nothing else."
            ),
            generate()
        ],
        scorer=match(ignore_case=True)
    )

eval(
    sentiment_classification,
    model="ollama/llama2"
)

Output()

##### Модель достигла accuracy 0.667 (4/6). Она корректно классифицировала все явно positive и negative примеры, но ошиблась на двух neutral примерах, оба раза отнеся их к positive. 
##### Это показывает, что в данной постановке локальная llama2 лучше распознаёт ярко выраженную эмоциональную окраску, чем нейтральные или пограничные высказывания. Ошибки были не в формате ответа, а в самой классификации.

In [16]:
# Сделаем еще прогон точнее объяснив модели, что считать neutral, 
# проверим, улучшится ли классификация neutral-примеров

@task
def sentiment_classification_v2():
    return Task(
        dataset=[
            Sample(
                input="I absolutely loved this movie, it was amazing.",
                target="positive"
            ),
            Sample(
                input="This was the worst restaurant experience of my life.",
                target="negative"
            ),
            Sample(
                input="The meeting is scheduled for 3 PM tomorrow.",
                target="neutral"
            ),
            Sample(
                input="The product works fine, nothing special but no major problems.",
                target="neutral"
            ),
            Sample(
                input="The customer support team was very helpful and kind.",
                target="positive"
            ),
            Sample(
                input="I am very disappointed with the quality of this phone.",
                target="negative"
            ),
        ],
        solver=[
            system_message(
                "You are a sentiment classifier. "
                "Classify the text into exactly one of these labels: positive, negative, neutral. "
                "Use 'neutral' for factual statements, objective descriptions, or mixed statements without a clear positive or negative opinion. "
                "Reply with only one label and nothing else."
            ),
            generate()
        ],
        scorer=match(ignore_case=True)
    )

eval(
    sentiment_classification_v2,
    model="ollama/llama2"
)

Output()

### Выводы: 

#### после уточнения system prompt качество изменилось: accuracy повысилось до 0.833, но модель снова ошиблась на тех же neutral-примерах. Это показывает, что в данном случае проблема не сводится к формулировке инструкции. Возможно локальная llama2 имеет устойчивую склонность интерпретировать нейтральные или слабо окрашенные высказывания как positive.

In [20]:
## ради интереса прогоним на другой модели, возьмем qwen2:7b-instruct-q4_K_M

eval(
    sentiment_classification_v2,
    model="ollama/qwen2:7b-instruct-q4_K_M"
)

Output()

##### На одном и том же sentiment classification task модель Llama2 показала accuracy 0.667 и ошиблась на двух neutral-примерах, относя их к positive. 
##### Уточнение system prompt немного помогло и повысило accuracy 0.833. Однако Qwen2 7B на той же задаче достигла accuracy 1.000. 
##### Это показывает, что различие объясняется не только формулировкой инструкции, но и возможно качеством самой модели: Qwen2 7B лучше различает neutral и positive в подобных текстах.

### 8.3. Single choice with explanation

Collect both choice and reasoning:

In [21]:
@task
def choice_with_reasoning():
    PROMPT = '''
Classify as True or False:

Statement: {prompt}

Provide:
1. REASONING: [Your explanation]
2. ANSWER: [True or False]
    '''.strip()

    return Task(
        dataset=[
            Sample(
                input="The Earth is flat.",
                target="False"
            ),
            Sample(
                input="Water boils at 100°C at sea level.",
                target="True"
            ),
        ],
        solver=[
            chain_of_thought(),
            prompt_template(PROMPT),
            generate()
        ],
        scorer=pattern(r'ANSWER:\s*(True|False)'),
    )

eval(choice_with_reasoning, model='ollama/llama2')

Output()

---
## Part 9: Multiple choice tasks

### 9.1. Understanding multiple choice in Inspect

Key rules:
- `choices`: list of answer options (no letters — they're added automatically)
- `target`: letter of correct answer ("A", "B", "C", or "D")
- Use `multiple_choice()` solver + `choice()` scorer

### 9.2. Multiple choice with metadata

Metadata lets you filter and analyze results in `inspect view`.

In [22]:
@task
def mc_with_metadata():
    return Task(
        dataset=[
            Sample(
                input="Capital of Japan?",
                choices=["Seoul", "Tokyo", "Bangkok", "Beijing"],
                target="B",
                metadata={
                    "difficulty": "easy",
                    "category": "geography"
                }
            ),
            Sample(
                input="What is the Heisenberg Uncertainty Principle?",
                choices=[
                    "Cannot know both position and momentum precisely",
                    "Energy cannot be created or destroyed",
                    "All matter has wave-particle duality",
                    "Time always moves forward"
                ],
                target="A",
                metadata={
                    "difficulty": "hard",
                    "category": "physics"
                }
            ),
        ],
        solver=multiple_choice(),
        scorer=choice(),
    )

# Run and check results in inspect view - filter by metadata!
eval(mc_with_metadata, model="ollama/llama2")

Output()

##### mc_with_metadata показывает, как строить multiple choice задачи в Inspect AI и как добавлять к sample дополнительную информацию через metadata. 
##### В этом прогоне Llama2 правильно решила оба примера, поэтому accuracy составила 1.0, но на таком маленьком наборе это скорее демонстрация корректной работы пайплайна, чем сильный вывод о качестве модели.

### 9.6. Multiple correct Answers

When multiple answers are valid:

In [23]:
@task
def mc_multiple_correct():
    return Task(
        dataset=[
            Sample(
                input="Which are programming languages?",
                choices=["Python", "HTML", "JavaScript", "CSS"],
                target=["A", "C"]  # Python, JavaScript
            ),
            Sample(
                input="Which continents border the Atlantic Ocean?",
                choices=["Africa", "Asia", "Europe", "South America"],
                target=["A", "C", "D"]  # Africa, Europe, South America
            ),
        ],
        solver=[
            system_message("Select ALL correct answers. You may choose multiple options."),
            multiple_choice(multiple_correct=True)
        ],
        scorer=choice(),
    )

eval(mc_multiple_correct, model="ollama/llama2")

Output()

##### В mc_multiple_correct модель должна выбрать не один, а несколько правильных вариантов.Первый sample не был засчитан, потому что модель выбрала все правильные ответы, но добавила лишний вариант D. Это показывает, что в multi-answer multiple choice задачах 
##### scorer требует точного совпадения множества ответов, а частично правильный ответ не считается correct.

##### Вывод: метрика зависит от того, что именно мы считаем “правильным”

---
## Part 10: Composing solvers together

## Quick reference

| Task type | Solvers | Scorer |
|-----------|---------|--------|
| Simple Q&A | `system_message() + generate()` | `match()` |
| Reasoning | `chain_of_thought() + generate()` | `match()` |
| Structured output | `prompt_template() + generate()` | `pattern()` |
| Classification | `system_message() + generate()` | `exact()` |
| Multiple choice | `multiple_choice()` | `choice()` |
| MC + reasoning | `multiple_choice(cot=True)` | `choice()` |

---
## Assignment 3: Analyzing position bias in multiple choice

Language models can develop **position bias** - a tendency to favor certain answer positions (like more often picking "A" or "C") regardless of content.

In this assignment, you will:
1. Generate a set of simple math questions in multiple-choice format
2. Create two versions of the dataset:
   - **Biased:** correct answer is always position A
   - **Unbiased:** correct answer position is randomized
3. Run evaluations on both and compare results
4. Analyze whether the model shows position bias

⚠️ **Note on methodology:** this is a minimal experiment to get you started. Comparing "all-A" vs "randomized" datasets is a quick sanity check, but it's not the most rigorous way to measure position bias.

Feel free to extend the assignment if you want a deeper analysis!

In [24]:
import random
from inspect_ai import Task, task, eval
from inspect_ai.dataset import Sample
from inspect_ai.scorer import choice
from inspect_ai.solver import multiple_choice, system_message

# For reproducibility
random.seed(42)

### Step 1: Generate questions

First, create a helper function that generates simple questions with known correct answers.

**Function spec:**
- Input: `n` (number of problems to generate)
- Output: list of tuples `(question_text, correct_answer)`
- Example output: `[("What is 5 + 3?", "8"), ("What is 12 - 4?", "8"), ...]`

**This is just an example.** You can implement your own generator with different content - trivia, vocabulary, geography, etc. Just make sure the model can reasonably answer them.

In [26]:
def generate_questions(n: int) -> list[tuple[str, int]]:
    """
    Generate n simple math problems.
    
    Args:
        n: number of problems to generate
        
    Returns:
        List of (question_text, correct_answer) tuples
    """
    problems = []
    
    for _ in range(n):
        x = random.randint(1, 20)
        y = random.randint(1, 20)
        op = random.choice(["+", "-"])
        
        if op == "+":
            answer = x + y
        else:
            answer = x - y
        
        question = f"What is {x} {op} {y}?"
        problems.append((question, str(answer)))
    
    return problems


test_questions = generate_questions(5)

assert len(test_questions) == 5, f"Expected 5 questions, got {len(test_questions)}"
assert all(isinstance(q, tuple) and len(q) == 2 for q in test_questions), "Each question must be a tuple of (question_text, answer)"
assert all(isinstance(q[0], str) and isinstance(q[1], str) for q in test_questions), "Both question and answer must be strings"
assert all(len(q[0]) > 0 and len(q[1]) > 0 for q in test_questions), "Question and answer cannot be empty"

print("\nSample output:")
for q, a in test_questions:
    print(f"  {q} → {a}")



Sample output:
  What is 8 + 17? → 25
  What is 18 - 7? → 11
  What is 8 - 15? → -7
  What is 1 - 6? → -5
  What is 11 + 9? → 20


### Step 2: Create wrong answers (distractors)

For multiple choice, we need plausible wrong answers.

**Function spec:**
- Input: `correct_answer` (int)
- Output: list of 3 wrong answers (ints), all different from correct and from each other

**Tip:** generate distractors close to the correct answer (e.g., ±1, ±2, ±10) to make them plausible.

In [30]:
def generate_distractors(correct: str, n: int = 3) -> list[str]:
    """
    Generate n plausible wrong answers.
    
    For numeric answers: generates nearby numbers.
    For other types: you'll need to customize this.
    
    Args:
        correct: the correct answer (string)
        n: number of distractors to generate
        
    Returns:
        List of n distinct wrong answers (strings)
    """
    distractors = set()

    offsets = [-3, -2, -1, 1, 2, 3, 4, -4]

    while len(distractors) < n:
    
    
        offset = random.choice(offsets)
        wrong = str(int(correct) + offset)
        if wrong != correct:
            distractors.add(wrong)
        
    return list(distractors)

test_distractors = generate_distractors("10", n=3)

assert len(test_distractors) == 3, f"Expected 3 distractors, got {len(test_distractors)}"
assert all(isinstance(d, str) for d in test_distractors), "All distractors must be strings"
assert "10" not in test_distractors, "Distractors must not include the correct answer"
assert len(set(test_distractors)) == 3, "All distractors must be unique"

print(f"   Distractors for '10': {test_distractors}")

   Distractors for '10': ['8', '6', '14']


### Step 3: Create multiple choice samples

Now create a function that converts questions into multiple-choice format.

**Function spec:**
- Input: 
  - `questions` - list of `(question_text, correct_answer)` tuples
  - `correct_position` - where to place correct answer:
    - `None` → randomize position for each question
    - `0` → always position A
    - `1` → always position B
    - `2` → always position C
    - `3` → always position D
- Output:
    - list of `Sample` objects

⚠️ **Note on `Sample` type:**

`Sample` is an Inspect AI class. For multiple choice, you create it like this:
```python
Sample(
    input="What is 2 + 2?",           # question text
    choices=["3", "4", "5", "6"],     # list of options: list[str]
    target="B"                         # letter of correct answer (A/B/C/D)
)
```

In [31]:
def create_samples(
    questions: list[tuple[str, int]], 
    correct_position: int | None = None
) -> list[Sample]:
    """
    Convert questions to multiple-choice Samples.
    
    Args:
        questions: list of (question_text, correct_answer) tuples
        correct_position: 
            None → randomize position (A/B/C/D) for each question
            0 → correct answer always at position A
            1 → correct answer always at position B
            2 → correct answer always at position C  
            3 → correct answer always at position D
            
    Returns:
        List of Sample objects ready for Inspect AI.
        Each Sample has:
            - input: str (the question)
            - choices: list[str] (4 options, no letters)
            - target: str (correct letter: "A", "B", "C", or "D")
    """
    samples = []
    letters = ["A", "B", "C", "D"]

    for question_text, correct_answer in questions:
        distractors = generate_distractors(correct_answer, n=3)

        if correct_position is None:
            correct_idx = random.randint(0, 3)
        else:
            correct_idx = correct_position

        choices = distractors.copy()
        choices.insert(correct_idx, correct_answer)

        samples.append(
            Sample(
                input=question_text,
                choices=choices,
                target=letters[correct_idx]
            )
        )

    return samples


### Step 4: Create the tasks

Now wrap your datasets into Inspect Tasks.

**Already provided:** task structure. You just need to call your functions.

In [32]:
@task
def position_bias_task(
    questions: list[tuple[str, int]],
    correct_position: int | None = None
):
    """
    Multiple choice evaluation task.
    
    Args:
        questions: list of (question_text, correct_answer) tuples
        correct_position: None for random, 0-3 for fixed position
    """
    samples = create_samples(questions, correct_position=correct_position)

    return Task(
        dataset=samples,
        solver=[multiple_choice()],
        scorer=choice()
    )

### Step 5: Generate questions and run evaluations

Generate questions once, then run two evaluations:
1. **Biased:** correct answer always at position A
2. **Unbiased:** correct answer position randomized

In [33]:
MODEL = "ollama/llama2"
N_QUESTIONS = 20

random.seed(42)
questions = generate_questions(N_QUESTIONS)

eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": 0}
)

eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": None}
)

Output()

Output()

In [34]:
MODEL = "ollama/qwen2:7b-instruct-q4_K_M"
N_QUESTIONS = 20

random.seed(42)
questions = generate_questions(N_QUESTIONS)

eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": 0}
)

eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": None}
)

Output()

Output()

#### В эксперименте по position bias модель Llama2 показала низкое качество как при фиксированной 
#### позиции правильного ответа A (accuracy 0.00), так и при случайной позиции (accuracy 0.25). 
#### Это не подтверждает гипотезу о предпочтении A и скорее указывает на слабую общую работу модели 
#### в данном multiple-choice setup. Модель Qwen2 7B показала существенно лучшие результаты: 
#### accuracy 0.80 при fixed A и 0.45 при random position. Такая разница может указывать на position bias, 
#### однако для более надёжного вывода нужно дополнительно сравнить все фиксированные позиции A/B/C/D.

In [ ]:
## пробуем еще эксперимент

MODEL = "ollama/qwen2:7b-instruct-q4_K_M"
N_QUESTIONS = 20

random.seed(42)
questions = generate_questions(N_QUESTIONS)

for pos, label in enumerate(["A", "B", "C", "D"]):
    print(f"\nRunning fixed position {label}...")
    eval(
        position_bias_task,
        model=MODEL,
        task_args={"questions": questions, "correct_position": pos}
    )

Output()


Running fixed position A...



Running fixed position B...


Output()


Running fixed position C...


Output()

Output()


Running fixed position D...


In [36]:
eval(
    position_bias_task,
    model=MODEL,
    task_args={"questions": questions, "correct_position": None}
)

Output()

#### Выводы:

##### В эксперименте по position bias модель Qwen2 7B показала наилучший результат, когда правильный ответ всегда находился в позиции A (accuracy 0.70). 
##### При фиксированных позициях B, C и D качество заметно ниже (0.35, 0.40 и 0.35 соответственно), а случайное распределение правильных ответов даёт промежуточный результат (0.45). 
##### Это указывает на вероятный position bias в сторону A: модель зависит не только от содержания вариантов ответа, но и от их расположения. Однако вывод следует считать предварительным, так как между прогонами могли меняться distractor’ы. 
##### Для более строгого эксперимента нужно зафиксировать сами варианты ответа и менять только их порядок.

### Step 7: Analyze your results

Open `inspect view` and examine both evaluation runs.

1. **Accuracy comparison:**
##### For Llama2
- Biased task accuracy: 0%
- Unbiased task accuracy: 25%

##### For Qwen2 7B
- Biased task accuracy: 70%
- Unbiased task accuracy: 45%
  
2. **Your analysis**
   
##### What do the numbers show? (just the facts)
- For Llama2, the model performed very poorly on both versions of the task. It got 0% accuracy when the correct answer was always in position A, and 25% when the correct answer position was randomized.
For Qwen2 7B, the model performed much better overall. It achieved 70% accuracy on the biased version and 45% on the unbiased version.

##### I also ran additional fixed-position experiments with Qwen2 7B:
  
- A fixed: 70%
- B fixed: 35%
- C fixed: 40%
- D fixed: 35%
- Random: 45%

##### What patterns do you notice?

For Llama2, I do not see evidence of a preference for position A. The model was weak overall and often selected wrong options regardless of where the correct answer was placed.
For Qwen2 7B, I noticed a clear pattern: performance was highest when the correct answer was in position A and lower when it was in B, C, or D. The random setting gave an intermediate result, which is also consistent with this pattern.
This suggests that Qwen2 7B may be influenced not only by the content of the options, but also by their position. 
 
##### What might explain them? What doesn't fit your explanation?

A possible explanation is position bias: the model may be more likely to select the first option, especially when it is uncertain. This would explain why Qwen2 7B does much better when the correct answer is always at A.
However, this explanation is not perfect. In my current setup, the distractors may also change between runs, so the experiment does not isolate position as the only changing variable. That means part of the difference could come from different wrong answer choices, not just from answer position.
For Llama2, the position-bias explanation does not fit well. Its results suggest that the bigger problem is low overall performance on this multiple-choice arithmetic task.

##### What else did you try, and what did you learn?

I compared two different models:
##### Llama2

##### Qwen2 7B

This was useful because it showed that the same benchmark can reveal different kinds of weaknesses:
Llama2 was weak overall and did not show a clear position preference.
Qwen2 7B was much stronger, and because of that, a more systematic pattern became visible.
I also ran extra fixed-position experiments for Qwen2 7B with answers always placed in A, B, C, and D. This helped me see that the effect was not just “biased vs random,” but more specifically a likely preference for A.
I learned that:
even a small eval can reveal useful behavioral patterns,
model quality and evaluation design both matter,
to make the conclusion stronger, I should keep the same distractors and only permute their order.
   

#### Short conclusion
##### My results suggest that Qwen2 7B likely shows position bias toward option A, while Llama2 mainly shows weak general performance on this task. 
##### At the same time, the experiment is still preliminary, because the distractors may vary across runs. A stronger follow-up would fix the answer choices and only change their positions.

### Bonus challenges (optional)

If you want to explore further:

1. **Try different models** - Do all models show the same bias pattern?

2. **Add Chain-of-Thought** - Does `multiple_choice(cot=True)` reduce position bias?

3. **More positions** - What if you have 5 or 6 choices instead of 4?

4. **Statistical test** - Is the position preference statistically significant? (Chi-squared test)

5. **Content factors** - What else might affect position bias?

In [39]:
## Add Chain-of-Thought - Does multiple_choice(cot=True) reduce position bias?

## Что мы проверяем
## Гипотеза:
## если модели дать chain-of-thought в multiple choice, станет ли она меньше зависеть от позиции ответа

@task
def position_bias_task_cot(
    questions: list[tuple[str, int]],
    correct_position: int | None = None
):
    """
    Multiple choice evaluation task with Chain-of-Thought enabled.
    
    Args:
        questions: list of (question_text, correct_answer) tuples
        correct_position: None for random, 0-3 for fixed position
    """
    samples = create_samples(questions, correct_position=correct_position)

    return Task(
        dataset=samples,
        solver=[multiple_choice(cot=True)],
        scorer=choice()
    )


In [40]:
## Проверяем на qwen2:7b-instruct-q4_K_M

MODEL = "ollama/qwen2:7b-instruct-q4_K_M"
N_QUESTIONS = 20

random.seed(42)
questions = generate_questions(N_QUESTIONS)

print("Running CoT fixed A...")
eval(
    position_bias_task_cot,
    model=MODEL,
    task_args={"questions": questions, "correct_position": 0}
)

print("\nRunning CoT random...")
eval(
    position_bias_task_cot,
    model=MODEL,
    task_args={"questions": questions, "correct_position": None}
)

Output()

Running CoT fixed A...


Output()


Running CoT random...


### Выводы:

#### Добавление Chain-of-Thought сильно повлияло на результаты Qwen2 7B. 
#### Без CoT модель показывала 70% accuracy на biased версии и 45% на unbiased, что указывало на возможный position bias в сторону A. 
#### С multiple_choice(cot=True) качество выросло до 90% на biased задаче и 95% на unbiased. Это означает, что CoT почти устранил разницу между режимами и, вероятно, уменьшил зависимость модели от позиции ответа. 
#### Наиболее правдоподобное объяснение состоит в том, что CoT заставил модель сначала реально решать арифметическую задачу, а уже потом выбирать соответствующий вариант.

In [41]:
## проведем эксперимент с увеличением количества ответов на выбор

def create_samples_n_choices(
    questions: list[tuple[str, str]], 
    num_choices: int = 4,
    correct_position: int | None = None
) -> list[Sample]:
    """
    Convert questions to multiple-choice Samples with variable number of choices.
    
    Args:
        questions: list of (question_text, correct_answer) tuples
        num_choices: total number of answer choices (e.g. 4, 5, 6)
        correct_position:
            None -> randomize correct answer position
            0..num_choices-1 -> fixed correct position
            
    Returns:
        List of Sample objects
    """
    samples = []
    letters = [chr(ord("A") + i) for i in range(num_choices)]

    for question_text, correct_answer in questions:
        distractors = generate_distractors(correct_answer, n=num_choices - 1)

        if correct_position is None:
            correct_idx = random.randint(0, num_choices - 1)
        else:
            correct_idx = correct_position

        choices = distractors.copy()
        choices.insert(correct_idx, correct_answer)

        samples.append(
            Sample(
                input=question_text,
                choices=choices,
                target=letters[correct_idx]
            )
        )

    return samples

In [42]:
@task
def position_bias_task_n_choices(
    questions: list[tuple[str, int]],
    num_choices: int = 4,
    correct_position: int | None = None
):
    """
    Multiple choice evaluation task with variable number of choices.
    
    Args:
        questions: list of (question_text, correct_answer) tuples
        num_choices: total number of answer choices
        correct_position: None for random, 0..num_choices-1 for fixed position
    """
    samples = create_samples_n_choices(
        questions,
        num_choices=num_choices,
        correct_position=correct_position
    )

    return Task(
        dataset=samples,
        solver=[multiple_choice()],
        scorer=choice()
    )

In [43]:
## попробуем увеличить количество ответов до 5

MODEL = "ollama/qwen2:7b-instruct-q4_K_M"
N_QUESTIONS = 20

random.seed(42)
questions = generate_questions(N_QUESTIONS)

print("Running 5-choice fixed A...")
eval(
    position_bias_task_n_choices,
    model=MODEL,
    task_args={
        "questions": questions,
        "num_choices": 5,
        "correct_position": 0
    }
)

print("\nRunning 5-choice random...")
eval(
    position_bias_task_n_choices,
    model=MODEL,
    task_args={
        "questions": questions,
        "num_choices": 5,
        "correct_position": None
    }
)

Output()

Running 5-choice fixed A...



Running 5-choice random...


Output()

In [44]:
## попробуем увеличить количество ответов до 6

print("Running 6-choice fixed A...")
eval(
    position_bias_task_n_choices,
    model=MODEL,
    task_args={
        "questions": questions,
        "num_choices": 6,
        "correct_position": 0
    }
)

print("\nRunning 6-choice random...")
eval(
    position_bias_task_n_choices,
    model=MODEL,
    task_args={
        "questions": questions,
        "num_choices": 6,
        "correct_position": None
    }
)

Output()

Running 6-choice fixed A...



Running 6-choice random...


Output()

#### Выводы:

##### При увеличении числа вариантов ответа с 4 до 5 и 6, Qwen2 7B всё равно модель показывала лучший результат в случае, когда правильный ответ был зафиксирован в позиции A, чем при случайном расположении. 

##### Разрыв сохранялся во всех настройках: 0.70 против 0.45 для 4 вариантов, 0.85 против 0.60 для 5 вариантов и 0.80 против 0.50 для 6 вариантов. 

##### Это указывает на то, что position bias в сторону A сохраняется и при большем числе вариантов ответа. Однако результаты не показывают чистого монотонного усиления bias, потому что между прогонами менялись distractor’ы, а значит менялась и общая сложность задачи.

In [ ]:
import sys
!{sys.executable} -m pip install scipy

In [46]:
## 4. Statistical test - Is the position preference statistically significant? (Chi-squared test)

from scipy.stats import chi2_contingency
import numpy as np

# Qwen2 7B results, 20 questions per condition
# rows = positions A, B, C, D
# columns = [correct, incorrect]
table = np.array([
    [14,  6],   # A fixed: 14 correct, 6 incorrect
    [ 7, 13],   # B fixed
    [ 8, 12],   # C fixed
    [ 7, 13],   # D fixed
])

chi2, p_value, dof, expected = chi2_contingency(table)

print("Contingency table:")
print(table)

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.6f}")

print("\nExpected counts under H0 (no position effect):")
print(expected)

Contingency table:
[[14  6]
 [ 7 13]
 [ 8 12]
 [ 7 13]]

Chi-square statistic: 6.8687
Degrees of freedom: 3
P-value: 0.076203

Expected counts under H0 (no position effect):
[[ 9. 11.]
 [ 9. 11.]
 [ 9. 11.]
 [ 9. 11.]]


#### Выводы:

##### Мы провели chi-squared test of independence для результатов Qwen2 7B при фиксированных позициях A/B/C/D. Тест дал χ² = 6.87, df = 3, p = 0.076. 

##### Это означает, что при стандартном уровне значимости 0.05 мы не можем уверенно утверждать, что зависимость между позицией правильного ответа и точностью модели статистически значима. Тем не менее, результаты показывают заметный тренд: модель лучше справлялась, когда правильный ответ находился в позиции A. Для более надёжного вывода нужно увеличить число вопросов и зафиксировать distractor’ы между прогонами.

#### 5. Content factors - What else might affect position bias?

##### На position bias могут влиять не только сами позиции ответов, но и другие факторы:

##### Во-первых, важны distractor’ы: если неправильные варианты слишком очевидны, модели легче решить задачу по содержанию, и bias может быть слабее; если distractor’ы правдоподобны, bias может усиливаться. 

##### Во-вторых, влияет сложность задачи: когда модель не уверена, она чаще опирается на поверхностные эвристики, включая позицию ответа. 

##### В-третьих, большое значение имеет способ elicitation — например, в моём эксперименте добавление Chain-of-Thought почти убрало разрыв между biased и unbiased условиями. Кроме того, на результат влияют число вариантов ответа, тип задачи, конкретная модель и размер датасета. Поэтому position bias нужно рассматривать не как изолированное свойство модели, а как эффект, возникающий из взаимодействия модели, задачи и дизайна eval.

#### Общие выводы по челленджу:

##### Мы провели несколько дополнительных экспериментов, чтобы лучше понять position bias. Сравнение моделей показало, что Llama2 и Qwen2 7B ведут себя по-разному: 
- у Llama2 не было явного bias toward A, но она в целом плохо справлялась с задачей
- у Qwen2 7B появился более выраженный эффект.
##### Добавление Chain-of-Thought резко улучшило результаты Qwen2 7B и почти убрало разрыв между biased и unbiased условиями, что говорит о снижении зависимости от позиционных эвристик. 

##### При увеличении числа вариантов ответа до 5 и 6 position effect сохранился, хотя не усиливался строго монотонно. 

##### Статистический тест (chi-squared) показал заметный тренд в сторону A, но не дал значимости на уровне 0.05. В целом bonus-эксперименты показывают, что position bias зависит не только от модели, но и от способа elicitation, числа вариантов ответа, distractor’ов и размера выборки.